# Notebook 02 — Feature Extraction
**DSP501 — Speaker Identification**

This notebook explains and demonstrates:
- What MFCCs are and why they work for speaker ID
- Extracting MFCC from one file (step by step)
- Building the full feature dataset for both pipelines
- Visualizing MFCC heatmaps (raw vs filtered)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

from preprocess import preprocess
from filter import design_fir, apply_filter
from feature_extraction import extract_mfcc, save_features

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 1. What is MFCC?

**Mel-Frequency Cepstral Coefficients** capture the shape of the vocal tract — which is unique per speaker.

Pipeline:
```
audio → framing → FFT → Mel filterbank → log → DCT → MFCCs
```

We use **13 coefficients**, then compute **mean + std** over time → 26-dim feature vector.

## 2. Load and preprocess one file

In [ ]:
SAMPLE = '../data/raw/speaker_01/01.wav'
SR = 16000

y_raw, sr = preprocess(SAMPLE, sr=SR)

coeffs = design_fir()
y_filt = apply_filter(y_raw, coeffs)

print(f'Audio shape: {y_raw.shape}')

## 3. Compute MFCC matrix (13 × time frames)

In [ ]:
mfcc_raw  = librosa.feature.mfcc(y=y_raw,  sr=SR, n_mfcc=13, n_fft=512, hop_length=256)
mfcc_filt = librosa.feature.mfcc(y=y_filt, sr=SR, n_mfcc=13, n_fft=512, hop_length=256)

print(f'MFCC matrix shape: {mfcc_raw.shape}  (13 coefficients × time frames)')

## 4. MFCC Heatmap — Raw vs Filtered

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

for ax, mfcc, title in zip(axes,
                            [mfcc_raw, mfcc_filt],
                            ['MFCC — Raw', 'MFCC — Filtered']):
    img = librosa.display.specshow(mfcc, sr=SR, hop_length=256,
                                   x_axis='time', ax=ax, cmap='coolwarm')
    ax.set_title(title)
    ax.set_ylabel('MFCC coefficient')
    fig.colorbar(img, ax=ax)

plt.tight_layout()
plt.savefig('../figures/mfcc_heatmap.png', dpi=150)
plt.show()

## 5. Aggregate: mean + std → 26-dim feature vector

In [ ]:
feat_raw  = extract_mfcc(y_raw)
feat_filt = extract_mfcc(y_filt)

print(f'Feature vector shape : {feat_raw.shape}')
print(f'First 13 (mean)      : {feat_raw[:13].round(2)}')
print(f'Last  13 (std)       : {feat_raw[13:].round(2)}')

## 6. Build full dataset for both pipelines

> Run this cell once. Saves `.npy` files to `features/`.

In [ ]:
X_raw, X_filt, y = save_features('../data/index.csv',
                                  features_dir='../features',
                                  data_dir='../data/raw')

print(f'\nPipeline A (raw)      X shape: {X_raw.shape}')
print(f'Pipeline B (filtered) X shape: {X_filt.shape}')
print(f'Labels                y shape: {y.shape}')
print(f'Unique speakers: {sorted(set(y.tolist()))}')

## 7. Feature distribution (first 2 MFCC means)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, X, title in zip(axes, [X_raw, X_filt], ['Pipeline A — Raw', 'Pipeline B — Filtered']):
    for spk in sorted(set(y.tolist())):
        mask = y == spk
        ax.scatter(X[mask, 0], X[mask, 1], label=f'Speaker {spk}', s=40)
    ax.set_title(title)
    ax.set_xlabel('MFCC-1 mean')
    ax.set_ylabel('MFCC-2 mean')
    ax.legend()

plt.tight_layout()
plt.savefig('../figures/feature_scatter.png', dpi=150)
plt.show()